In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.chdir("..")

In [224]:
from aether.clause.nodes.nodes import SMA
from aether.clause.nodes.nodes import ADD
from aether.clause.nodes.nodes import SHIFT
from aether.clause.nodes.nodes import DIFF
from aether.clause.nodes.nodes import PctChange
from aether.clause.nodes.nodes import ShiftSign
from aether.clause.nodes.nodes import ABS
from aether.clause.nodes.nodes import DIV
from aether.clause.nodes.nodes import SUB
from aether.clause.nodes.nodes import Comparison
from aether.clause.nodes.nodes import NewHigh
from aether.clause.nodes.nodes import NewLow
from aether.clause.nodes.nodes import ZSCORE
from aether.clause.nodes.nodes import MAX
from aether.clause.nodes.nodes import MIN
from aether.clause.nodes.nodes import STD
from aether.clause.nodes.nodes import SKEW
from aether.clause.nodes.nodes import KURT
from aether.clause.nodes.nodes import DATA
from aether.clause.nodes.nodes import LargerThan
from aether.clause.nodes.nodes import SmallerThan
from aether.clause.nodes.nodes import UpStreak
from aether.clause.nodes.nodes import DownStreak
from aether.clause.nodes.nodes import MeanRevertKick
from aether.clause.nodes.nodes import PullbackWithinBand
from aether.clause.nodes.nodes import DrawdownExceed
from aether.clause.nodes.nodes import JumpDetect
from aether.clause.nodes.nodes import ZBetween
from aether.clause.nodes.nodes import EqualApprox
from aether.clause.nodes.nodes import ZEXP
from aether.clause.nodes.nodes import ZSigmoid
from aether.clause.nodes.nodes import CrossDown
from aether.clause.nodes.nodes import CrossUp
from aether.clause.nodes.nodes import SlopeSignChange

from aether.clause.tree import ClauseTree
from aether.clause.tree.generator import ClauseGenerator
from aether.clause.graph.extractor import SubgraphExtractor
from aether.clause.graph.edge import SUEdgeCalculator
from aether.clause.graph.base import ClauseGraph
from aether.utils import generate_uuid

In [12]:
# 모든 노드들의 인스턴스를 포함한 리스트
nodes = [
    # 기본 수학 연산 노드들
    ADD(),
    DIV(), 
    SUB(),
    ABS(),
    SMA(period=10),
    SHIFT(period=10),
    DIFF(period=10),
    PctChange(period=10),
    STD(period=10),
    NewHigh(period=10),
    NewLow(period=10),
    MAX(period=10),
    MIN(period=10),
    ZSCORE(period=10), 
    SKEW(period=10),
    KURT(period=10),
    ZEXP(period=10),
    ZSigmoid(period=10),

    # 루트 노드들 
    Comparison(),
    ZBetween(period=10, lo=-1.5, hi=1.5),
    EqualApprox(tol=1e-2),
    CrossUp(),
    CrossDown(),
    SlopeSignChange(p=10),
    UpStreak(p=10),
    DownStreak(p=10),
    MeanRevertKick(p=10, z_th=1.0, dmax=10, eps=0.01),
    PullbackWithinBand(p=10, k=0.5),
    DrawdownExceed(pct=0.1, lookback=10),
    JumpDetect(p=10, q_tail=0.1),
    
    # 데이터 노드들 (label 파라미터 필요)
    DATA("OPEN", "BTCUSDT"),
    DATA("HIGH", "BTCUSDT"), 
    DATA("LOW", "BTCUSDT"),
    DATA("CLOSE", "BTCUSDT"),
    DATA("VOLUME", "BTCUSDT"),
]

In [13]:
generator = ClauseGenerator(nodes)

In [14]:
tree = generator.generate(max_depth=3)

tree.iscompleted, tree.depth

(False, 13)

In [171]:
trees = []

while len(trees) < 2:
    tree = generator.generate(max_depth=3)

    if tree.depth == 3 and tree.iscompleted:
        trees.append(tree)

In [172]:
tree_a, tree_b = trees

In [173]:
print(tree_a)

print(tree_b)

SlopeSignChange(10)
└── ABS()
    └── DIFF(10)
        └── DATA[OPEN]
CrossDown()
├── STD(10)
│   └── ZSCORE(10)
│       └── DATA[OPEN]
└── ABS()
    └── ZSigmoid(10)
        └── DATA[HIGH]


In [174]:
result_a = tree_a.evaluate()

result_b = tree_b.evaluate()

In [175]:
result_a.value_counts()

OPEN
False    33652
True      5732
Name: count, dtype: int64

In [176]:
result_b.value_counts()

False    37200
True      2184
Name: count, dtype: int64

In [177]:
SUEdgeCalculator.calculate(
    series_a = result_a,
    series_b = result_b,
    alpha = 0.0,
)

{'H_A': 0.414896479899421,
 'H_B': 0.21427130409518746,
 'H_AB': 0.6291670434272415,
 'MI': 7.405673669858004e-07,
 'SU': 2.3541172508354194e-06,
 'T': 39384,
 'counts': (314, 5418, 1870, 31782),
 'p_table': {(1, 1): 0.007972780824700387,
  (1, 0): 0.13756855575868374,
  (0, 1): 0.04748121064391631,
  (0, 0): 0.8069774527726996}}

## Graph

In [189]:
# 2. Generate Multiple ClauseTrees
trees = []
n_trees = 8
depth = 3

while len(trees) < n_trees:

    tree = generator.generate(max_depth=depth)
    tree_id = generate_uuid()
    tree.name = tree_id

    if tree.iscompleted and tree.depth == depth:
        trees.append(tree)

In [198]:
clause_graph = ClauseGraph(name = "Graph")
clause_graph.set_edge_calculator(SUEdgeCalculator.calculate)

2025-09-27 23:30:56 - aether.clause.graph.base - INFO - Edge calculator set


In [200]:
ids = clause_graph.add_clause_trees(trees)

2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: 0a7e39d0
2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: 55a9f902
2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: 78990979
2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: e0aaa0df
2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: 2879dd62
2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: 21feef40
2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: a8e47f67
2025-09-27 23:31:19 - aether.clause.graph.base - INFO - Added ClauseTree as node id: e5fbe65e


In [206]:
clause_graph.compute_all_edges()

2025-09-28 00:19:03 - aether.clause.graph.base - INFO - Added 28 edges with threshold 0.0


In [209]:
clause_graph.num_nodes

8

In [208]:
clause_graph.num_edges

28

In [212]:
clause_graph

ClauseGraph(8 nodes)

In [213]:
clause_graph.clause_trees

{'0a7e39d0': <aether.clause.tree.base.ClauseTree at 0x12fd57750>,
 '55a9f902': <aether.clause.tree.base.ClauseTree at 0x139770210>,
 '78990979': <aether.clause.tree.base.ClauseTree at 0x1382b3990>,
 'e0aaa0df': <aether.clause.tree.base.ClauseTree at 0x1397da690>,
 '2879dd62': <aether.clause.tree.base.ClauseTree at 0x139762910>,
 '21feef40': <aether.clause.tree.base.ClauseTree at 0x1397a3c90>,
 'a8e47f67': <aether.clause.tree.base.ClauseTree at 0x13974bcd0>,
 'e5fbe65e': <aether.clause.tree.base.ClauseTree at 0x1397b1090>}

In [227]:
clause_graph.get_neighbors(node_id = '78990979')

['0a7e39d0',
 '55a9f902',
 'e0aaa0df',
 '2879dd62',
 '21feef40',
 'a8e47f67',
 'e5fbe65e']

In [243]:
extractor = SubgraphExtractor(clause_graph)

In [244]:
subgraph = extractor.extract(size = 3, start_node = '78990979')

2025-09-28 22:33:00 - aether.clause.graph.extractor - INFO - Random walk completed. Final size: 3
